In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
from dotenv import load_dotenv
from sqlalchemy import create_engine
import lightgbm as lgb
from lifelines import KaplanMeierFitter
import shap
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings('ignore')


load_dotenv(dotenv_path='../.env')

print("Initializing Secure Risk Engine...")

DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}")

print("Extracting daily matrix from MySQL...")
query = """
    SELECT * FROM vw_revenue_retention_matrix
    ORDER BY account_id, observation_date
"""
df = pd.read_sql(query, engine)
df['observation_date'] = pd.to_datetime(df['observation_date'])
print(f"Ingested {len(df)} daily rows representing {df['account_id'].nunique()} unique accounts.")

Initializing Secure Risk Engine...
Extracting daily matrix from MySQL...
Ingested 122771 daily rows representing 500 unique accounts.


In [ ]:
print("Engineering T-1 temporal observation windows...")

churn_dates = df[df['target_churn_flag'] == 1].groupby('account_id')['observation_date'].max().reset_index()
churn_dates.rename(columns={'observation_date': 'exit_date'}, inplace=True)
df = df.merge(churn_dates, on='account_id', how='left')

max_date = df['observation_date'].max()

df['eval_end_date'] = df['exit_date'].apply(lambda x: x - pd.Timedelta(days=1) if pd.notnull(x) else max_date)
df['eval_start_date'] = df['eval_end_date'] - pd.Timedelta(days=30)

critical_window = df[(df['observation_date'] > df['eval_start_date']) & 
                     (df['observation_date'] <= df['eval_end_date'])]

critical_window['is_last_7_days'] = critical_window['observation_date'] > (critical_window['eval_end_date'] - pd.Timedelta(days=7))

feature_store = critical_window.groupby('account_id').agg({
    'industry': 'first',
    'baseline_tier': 'first',
    'referral_source': 'first',
    'billing_frequency': 'first',
    'active_mrr_run_rate': 'last',  
    'net_mrr_delta': 'sum',         
    'downgrade_flag': 'max',
    'auto_renew_flag': 'max',
    'is_trial': 'max',
    'daily_time_in_app': 'mean',    
    'daily_friction_events': 'sum', 
    'ticket_volume': 'sum',         
    'avg_resolution_time': 'mean',
    'avg_csat_score': 'mean',
    'eval_end_date': 'max'
}).reset_index()

last_7d = critical_window[critical_window['is_last_7_days']].groupby('account_id')['daily_time_in_app'].mean()
prev_23d = critical_window[~critical_window['is_last_7_days']].groupby('account_id')['daily_time_in_app'].mean()
momentum = (last_7d / (prev_23d + 1)).fillna(1.0).reset_index(name='app_usage_momentum')

feature_store = feature_store.merge(momentum, on='account_id', how='left')

target_df = df.groupby('account_id')['target_churn_flag'].max().reset_index()
feature_store = feature_store.merge(target_df, on='account_id')

feature_store['industry'] = feature_store['industry'].astype('category')
feature_store['baseline_tier'] = feature_store['baseline_tier'].astype('category')
feature_store['referral_source'] = feature_store['referral_source'].astype('category')
feature_store['billing_frequency'] = feature_store['billing_frequency'].astype('category')

Engineering T-1 temporal observation windows...


In [ ]:
print("Initializing LightGBM Classification Engine...")

cohort_mapping = df.groupby('account_id')['observation_date'].min().reset_index(name='first_seen_date')
feature_store = feature_store.merge(cohort_mapping, on='account_id')

feature_store = feature_store.sort_values('first_seen_date')
split_idx = int(len(feature_store) * 0.8)

train_data = feature_store.iloc[:split_idx]
test_data = feature_store.iloc[split_idx:]

drop_cols = ['account_id', 'target_churn_flag', 'eval_end_date', 'first_seen_date']

X_train = train_data.drop(columns=drop_cols)
y_train = train_data['target_churn_flag']
X_test = test_data.drop(columns=drop_cols)
y_test = test_data['target_churn_flag']

print(f"OOT Split Complete. Train Churn: {y_train.mean()*100:.1f}% | Test Churn: {y_test.mean()*100:.1f}%")

lgb_model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    class_weight='balanced',
    learning_rate=0.05,
    n_estimators=200,
    max_depth=5,       
    random_state=42
)

lgb_model.fit(X_train, y_train, categorical_feature=['industry', 'baseline_tier', 'referral_source', 'billing_frequency'])

predictions = lgb_model.predict(X_test)
probabilities = lgb_model.predict_proba(X_test)[:, 1]

print("\n--- LightGBM Out-of-Sample Performance ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, probabilities):.4f}")
print(classification_report(y_test, predictions))

Initializing LightGBM Classification Engine...
OOT Split Complete. Train Churn: 70.2% | Test Churn: 71.0%
[LightGBM] [Info] Number of positive: 281, number of negative: 119
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000145 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 383
[LightGBM] [Info] Number of data points in the train set: 400, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

In [ ]:
print("Computing Non-Parametric Survival Probabilities...")

survival_df = df.groupby('account_id').agg({
    'observation_date': ['min', 'max'],
    'target_churn_flag': 'max'
}).reset_index()
survival_df.columns = ['account_id', 'start_date', 'end_date', 'event_observed']
survival_df['duration_days'] = (survival_df['end_date'] - survival_df['start_date']).dt.days

kmf = KaplanMeierFitter()
kmf.fit(durations=survival_df['duration_days'], event_observed=survival_df['event_observed'])

print("Kaplan-Meier survival curve fitted.")

Computing Non-Parametric Survival Probabilities...
Kaplan-Meier survival curve fitted.


In [ ]:
print("Extracting SHAP interaction values...")

explainer = shap.TreeExplainer(lgb_model)

active_accounts = feature_store[feature_store['target_churn_flag'] == 0].copy()
X_active = active_accounts.drop(columns=['account_id', 'target_churn_flag', 'eval_end_date', 'first_seen_date'])

active_accounts['flight_risk_score'] = lgb_model.predict_proba(X_active)[:, 1]

shap_values = explainer.shap_values(X_active)

if isinstance(shap_values, list):
    shap_values = shap_values[1] 

feature_names = X_active.columns
top_drivers = []

for i in range(len(shap_values)):
    sorted_indices = np.argsort(shap_values[i])[::-1]
    driver_1 = feature_names[sorted_indices[0]]
    driver_2 = feature_names[sorted_indices[1]]
    
    top_drivers.append(f"1. {driver_1.replace('_', ' ').title()} | 2. {driver_2.replace('_', ' ').title()}")

active_accounts['primary_risk_drivers'] = top_drivers

Extracting SHAP interaction values...


In [ ]:
print("Serializing artifacts to the Application and Data layers...")

APP_DIR = '../app/'
DATA_DIR = '../data/'

joblib.dump(lgb_model, f'{APP_DIR}ravenstack_lgbm_risk_engine.pkl')
joblib.dump(kmf, f'{APP_DIR}ravenstack_survival_model.pkl')

# CALCULATE TRUE TENURE 
account_starts = df.groupby('account_id')['observation_date'].min().reset_index()
account_starts.rename(columns={'observation_date': 'signup_date'}, inplace=True)

active_accounts = active_accounts.merge(account_starts, on='account_id')

current_eval_date = df['observation_date'].max()
active_accounts['tenure_days'] = (current_eval_date - active_accounts['signup_date']).dt.days

final_output = active_accounts[[
    'account_id', 'industry', 'active_mrr_run_rate', 
    'flight_risk_score', 'primary_risk_drivers', 'tenure_days'  
]]

final_output.sort_values(by='flight_risk_score', ascending=False, inplace=True)

final_output.to_csv(f'{DATA_DIR}live_flight_risk_scores.csv', index=False)

print(f"SUCCESS: Models routed to {APP_DIR} and scoring matrix routed to {DATA_DIR}")

Serializing artifacts to the Application and Data layers...
SUCCESS: Models routed to ../app/ and scoring matrix routed to ../data/
